In [1]:
!pip install -q groq langchain-huggingface langchain-qdrant sentence-transformers python-dotenv


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.3/138.3 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 377.2/377.2 kB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 115.5 MB/s eta 0:00:00


In [2]:
import json, os, random, time, re
from pathlib import Path

from groq import Groq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore

In [ ]:
GROQ_API_KEY   = ""
QDRANT_URL     = ""
QDRANT_API_KEY = ""

# Qdrant collection used for similarity search (fastest model, no GPU needed)
COLLECTION_NAME = "ada_model_all_mini"
EMBED_MODEL     = "sentence-transformers/all-MiniLM-L6-v2"

# Path to your uploaded chunks folder
CHUNKS_DIR  = Path("/content/processed_chunks")
OUTPUT_PATH = Path("/content/eval_queries_multi_chunk.json")

NUM_QUERIES      = 30   # how many queries to generate
MIN_CHUNK_CHARS  = 300  # skip short / header chunk

In [4]:
groq_client = Groq(api_key=GROQ_API_KEY)
GROQ_MODEL  = "llama-3.3-70b-versatile"

TARGET_FILES = [
    "ada_chapter2_chunks.json",
    "ada_chapter3_chunks.json",
    "ada_chapter7_chunks.json",
    "ada_chapter13_chunks.json",
    "ada_cardio_disease_manag_chunks.json",
    "ada_chronic_kidney_manag_chunks.json",
    "ada_glycemic_hypoglycemia_chunks.json",
    "ada_obesity_and_weight_chunks.json",
    "ada_pharma_approaches_chunks.json",
    "ada_retino_neuro_footcare_chunks.json",
]


In [7]:
# ── Step 1: Connect to Qdrant (no local GPU — embeddings run via API) ─────────
print("Connecting to Qdrant...")
embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True},
)
vector_store = QdrantVectorStore.from_existing_collection(
    collection_name=COLLECTION_NAME,
    embedding=embeddings,
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
)
print("✓ Connected to Qdrant\n")


Connecting to Qdrant...
✓ Connected to Qdrant



In [ ]:
# ── Step 2: Load chunks from uploaded JSON files ──────────────────────────────
print("Loading chunks...")
all_chunks = []
chunk_id_to_text = {}

for filename in TARGET_FILES:
    fpath = CHUNKS_DIR / filename
    if not fpath.exists():
        print(f"{filename} not found — skipping")
        continue
    with open(fpath, "r", encoding="utf-8") as f:
        data = json.load(f)
    for chunk in data:
        text = chunk["page_content"].strip()
        if (
            len(text) >= MIN_CHUNK_CHARS
            and not text.startswith("Diabetes Care Volume")
            and "doi.org" not in text[:80]
            and not text.startswith("|")
        ):
            all_chunks.append(chunk)
            chunk_id_to_text[chunk["metadata"]["chunk_id"]] = text

print(f"✓ {len(all_chunks)} usable chunks from {len(TARGET_FILES)} files\n")

Loading chunks...
✓ 1146 usable chunks from 10 files



In [9]:
def find_related(seed_text: str, k: int = 7) -> list[dict]:
    """Qdrant similarity search — returns list of chunk dicts."""
    docs = vector_store.similarity_search(seed_text, k=k)
    return [{"page_content": d.page_content, "metadata": d.metadata} for d in docs]


def generate_query_and_scores(seed: dict, related: list[dict]) -> dict | None:
    """Ask Groq to write a multi-chunk nursing query and score each chunk."""
    seed_id = seed["metadata"]["chunk_id"]
    related  = [c for c in related if c["metadata"]["chunk_id"] != seed_id][:6]

    chunks_text  = f"SEED CHUNK:\n{seed['page_content'][:800]}\n\nRELATED CHUNKS:\n"
    chunks_text += "\n".join(
        f"[Chunk {i+1}]\n{c['page_content'][:500]}"
        for i, c in enumerate(related)
    )

    prompt = f"""You are a clinical nursing education expert creating retrieval evaluation questions.

Given these excerpts from ADA diabetes guidelines:

{chunks_text}

Tasks:
1. Write ONE nursing question that REQUIRES 2-3 of these chunks to answer fully.
   - Must be a real clinical scenario: medication decision, patient teaching, or safety concern.
   - Be specific (not "what is diabetes?").

2. Score each chunk's relevance:
   3 = Essential (directly answers the question)
   2 = Relevant (important supporting info)
   1 = Somewhat relevant
   0 = Not relevant

Return ONLY valid JSON (no markdown, no extra text):
{{
  "query": "...",
  "category": "factual|procedural|clinical_decision",
  "seed_relevance": <0-3>,
  "related_relevance": [<score1>, <score2>, ..., <score for each related chunk>]
}}"""

    try:
        resp = groq_client.chat.completions.create(
            model=GROQ_MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7,
            max_tokens=300,
        )
        raw = resp.choices[0].message.content.strip()
        # Strip markdown fences if present
        raw = re.sub(r"^```(?:json)?\s*", "", raw)
        raw = re.sub(r"\s*```$", "", raw)
        result = json.loads(raw)

        if not all(k in result for k in ["query", "category", "seed_relevance", "related_relevance"]):
            return None
        if len(result["related_relevance"]) != len(related):
            return None
        return result, related   # return related too so scores align
    except Exception as e:
        print(f"    ⚠️  LLM/parse error: {e}")
        return None


In [10]:
# ── Step 4: Main generation loop ──────────────────────────────────────────────
queries     = []
used_ids    = set()
attempts    = 0
max_attempts = NUM_QUERIES * 4

print("=" * 65)
print(f"Generating {NUM_QUERIES} multi-chunk queries")
print("=" * 65)

while len(queries) < NUM_QUERIES and attempts < max_attempts:
    attempts += 1

    seed = random.choice(all_chunks)
    seed_id = seed["metadata"]["chunk_id"]
    if seed_id in used_ids:
        continue
    used_ids.add(seed_id)

    print(f"\n[{len(queries)+1}/{NUM_QUERIES}] seed={seed_id} | {seed['metadata'].get('source','?')}")

    # Find related via Qdrant
    related_raw = find_related(seed["page_content"], k=7)
    related_raw = [c for c in related_raw if c["metadata"]["chunk_id"] != seed_id][:6]

    if len(related_raw) < 3:
        print("  ⚠️  Not enough related chunks")
        continue

    # Generate query
    out = generate_query_and_scores(seed, related_raw)
    if out is None:
        continue
    result, related = out

    print(f"  ✓ {result['query'][:80]}...")
    print(f"    category={result['category']}")

    # Build relevance map — only include chunks with score > 0
    relevance_scores = {}
    relevant_ids     = []

    if result["seed_relevance"] > 0:
        relevance_scores[seed_id] = result["seed_relevance"]
        if result["seed_relevance"] >= 2:
            relevant_ids.append(seed_id)

    for i, chunk in enumerate(related):
        cid   = chunk["metadata"]["chunk_id"]
        score = result["related_relevance"][i]
        if score > 0:
            relevance_scores[cid] = score
            if score >= 2:
                relevant_ids.append(cid)

    # Quality gate: need ≥2 high-relevance chunks
    if len(relevant_ids) < 2:
        print(f"  ⚠️  Only {len(relevant_ids)} high-relevance chunk(s) — skipping")
        continue

    print(f"    high-relevance chunks: {len(relevant_ids)} | total scored: {len(relevance_scores)}")

    queries.append({
        "question_id": f"Q{len(queries)+1:03d}",
        "query": result["query"],
        "category": result["category"],
        "ground_truth": {
            "relevant_chunk_ids": relevant_ids,
            "relevance_scores": relevance_scores,
            "seed_chunk_id": seed_id,
        },
        "metadata": {
            "seed_source": seed["metadata"].get("source", "unknown"),
        },
    })

    time.sleep(2)   # Groq free-tier rate limit

Generating 30 multi-chunk queries

[1/30] seed=4_102 | ada_chapter2.pdf
  ✓ A 55-year-old patient with advanced breast cancer is prescribed alpelisib and is...
    category=clinical_decision
    high-relevance chunks: 3 | total scored: 5

[2/30] seed=0_118 | ada_cardio_disease_manag.pdf
  ✓ What patient education should be provided to an individual with type 1 diabetes ...
    category=clinical_decision
    high-relevance chunks: 4 | total scored: 6

[3/30] seed=15_85 | ada_retino_neuro_footcare.pdf

[3/30] seed=15_106 | ada_retino_neuro_footcare.pdf
  ✓ A 30-year-old patient with type 1 diabetes, 20 weeks pregnant, presents with pro...
    category=clinical_decision
    high-relevance chunks: 6 | total scored: 7

[4/30] seed=14_193 | ada_pharma_approaches.pdf
  ✓ A 45-year-old patient with a history of type 2 diabetes is scheduled for organ t...
    category=clinical_decision
    high-relevance chunks: 5 | total scored: 7

[5/30] seed=1_26 | ada_chapter13.pdf
  ✓ An 80-year-old patien

In [11]:
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(queries, f, indent=2, ensure_ascii=False)

print(f"{len(queries)} queries saved to {OUTPUT_PATH.name}")

categories = {}
for q in queries:
    categories[q["category"]] = categories.get(q["category"], 0) + 1
avg_chunks = sum(len(q["ground_truth"]["relevance_scores"]) for q in queries) / max(len(queries), 1)

print("\nBreakdown:")
for cat, n in sorted(categories.items()):
    print(f"  {cat:22s}: {n}")
print(f"  Avg chunks per query  : {avg_chunks:.1f}")


30 queries saved to eval_queries_multi_chunk.json

Breakdown:
  clinical_decision     : 30
  Avg chunks per query  : 6.5

⬇️  Download eval_queries_multi_chunk.json from the Files panel on the left.
